# **PONTIFICIA UNIVERSIDAD JAVERIANA**
## **Procesamiento de alto volumen de datos**
**Fecha:** 7 de Abril del 2026

**Autor:** Grupo Sigma

**Tema:** Proyecto de Big Data

**Objetivo:** 
- Entender la importancia del uso de herramientas de Big Data en entornos empresariales, a fin de poder solucionar preguntas de negocio.
- Entender el paso a paso de un proyecto de procesamiento de datos para la generación de hallazgos de valor basado en la metodología CRISP-DM.
- Documentar la implementación de un cluster como infraestructura de procesamiento de grandes volúmenes de datos, a través de máquinas virtuales
- Realizar procesamiento de datos aplicado a entender y mejorar los indicadores ICFES en el territorio nacional.
- Integrar conjuntos de datos territoriales, sociales y económicos con los resultados educativos nacionales.
- Realizar exploración y transformación del conjunto global de datos disponible que permita su análisis de forma correcta y sin sesgos.


**Version:** Entrega 1

### Para asegurar que el proyecto funcione correctamente con pandas, matplotlib, seaborn y findspark, ejecutar el siguiente comando desde la raíz del proyecto
```bash
pip install -r requirements.txt
```

In [3]:
### Importación de bibliotecas basicas 
import os                       # -> Para gestion de archivos y procesos
import sys                      # -> Para manejo de recursos del sistema
import pandas as pd             # -> Para graficar y objetos dataframe
import numpy as np              # -> Para algebra matricial
import matplotlib.pyplot as plt # -> Para formatos de graficas
import seaborn as sns           # -> Para estadistica y graficar
import scipy.stats as stats     # -> Para pruebas estadisticas

In [4]:
### Importacion de bibliotecas especializadas
import findspark                                # -> Para manejo del entorno de PySpark
findspark.init('/Almacen/Spark')                # -> Se inicia el entorno para PySpark
from pyspark import SparkConf, SparkContext     # -> Para contexto y configuración de PySpark
from pyspark.sql import SparkSession            # -> Para manejo de Sesion en entorno de consultas SQL
from pyspark.sql import functions as F          # -> Para funciones de manipulacion de columnas
from pyspark.sql.types import IntegerType, StringType, DoubleType # -> Para definir tipos de datos
from pyspark.ml.feature import VectorAssembler  # -> Para construcción de vectores  
from pyspark.ml.stat import Correlation         # -> Para calculo de correlaciones

In [5]:
configura = SparkConf()
configura.set('spark.scheduler.mode', 'FAIR')
configura.set('spark.scheduler.allocation','/Almacen/Spark/conf/fairscheduler.xml')
configura.setMaster('spark://10.43.97.166:7077')
configura.setAppName('SigmaSPARK')

sparkSigma = SparkSession.builder.config(conf=configura).getOrCreate()
sparkSigma

### Actividades a realizar

- Lectura de data
- Descripción de los datos
- Exploración de los datos
- Reporte de calidad
- Planteamiento de preguntas investigativas
- Limpieza, filtro y transformaciones iniciales

## **Lectura de data**

In [6]:
## Se crea el dataframe para acceder al sistema de fichero csv como un objeto dataframe pyspark
## El acceso se hara desde la carpeta data del proyecto, donde se encuentran las bases de datos seleccionadas
dfPy00 = sparkSigma.read.format("csv").option("header","true").load("../data/Resultados_únicos_Saber_11_20260408.csv")
dfPy00.show(5)

+-------+------------------+----------------+-------------------+-------------+---------------+-----------------+-----------------------------+------------------+------------------------+------------------------+-----------------+--------------------+-----------+------------+--------------------+---------------+---------------------------+----------------+-------------------+---------------------------+---------------------------+---------------------+---------------------+-----------------------+-----------------+------------------------+---------------+--------------------+-----------+-----------------------+-----------------+-----------------+----------------+---------------------+-----------------+--------------------+--------------------+--------------------+------------------+-------------------+--------------------+------------------+------------------+-------------+-----------+----------------+------------------------+----------------+--------------------+-----------+
|PERIODO|

## **Descripción de los datos**

In [ ]:
dfPy00.printSchema()

Este conjunto de datos recopila métricas sobre el impacto del COVID-19 a nivel municipal. Su propósito es centralizar las tasas de incidencia y mortalidad del COVID-19, segmentada por cada uno de los municipios de Colombia, permitiendo análisis comparativos y geográficos de la propagación del virus y su letalidad. El cual en el contexto del proyecto, nos puede permitir entender el efecto de los problemas sanitarios en el rendimiento de los estudiantes en las pruebas del ICFES 11

A continuación, se detalla el significado de cada variable dentro del dataset:

- ***codigo_dane***: Identificador numérico único asignado por el Departamento Administrativo Nacional de Estadística (DANE) a cada municipio. Es la llave principal para cruces de datos con otras bases oficiales.
- ***municipio***: Nombre geográfico de la entidad territorial a la que pertenecen los datos.
- ***tasa_incidencia***: Representa la cantidad de casos positivos confirmados de COVID-19 por cada **100,000 habitantes**. Es un indicador de la propagación del virus en la población.
- ***tasa_mortalidad***: Indica la cantidad de fallecimientos atribuidos al COVID-19 por cada **1,000,000 de habitantes**. Mide el impacto de la enfermedad en términos de pérdida de vidas.

Diccionario de datos:
| Nombre de la Variable | Tipo de Dato | Definición | Ejemplo |
| :--- | :--- | :--- | :--- |
| `codigo_dane` | **String** | Código numérico único oficial del DANE para identificar municipios. Se maneja como texto para conservar el cero inicial. | `"05002"` |
| `municipio` | **String** | Nombre de la entidad territorial municipal. | `"ABEJORRAL"` |
| `tasa_incidencia` | **Double** | Cantidad de casos positivos confirmados por cada **100.000 habitantes**. | `2909.14` |
| `tasa_mortalidad` | **Double** | Cantidad de fallecimientos confirmados por cada **1.000.000 de habitantes**. | `772.29` |

In [ ]:
## Se cambia el tipo de dato de la base a la estipulada
# VERSION
# Aplicando el cambio de tipo de datos
dfPy01 = dfPy00.withColumn("tasa_incidencia", col("tasa_incidencia").cast(DoubleType())) \
              .withColumn("tasa_mortalidad", col("tasa_mortalidad").cast(DoubleType()))

# Verificar el resultado
dfPy01.printSchema()

## **Exploración de los datos**

In [ ]:
# Estadísticas descriptivas para las variables numéricas
dfPy01.select("tasa_incidencia", "tasa_mortalidad").summary().show()

Se evidencian que de todos los municipios de Colombia, en promedio se obtuvo una tasa de 4747.84 personas confirmadas por COVID-19 por cada 100 mil habitantes y una tasa de 1311.28 personas fallecidas por cada millón de habitantes. Con una tasa de incidencia y mortalidad mínima de 0 personas y una tasa máxima de 30251.44 personas confirmadas por COVID-19 y de 7644.32 personas fallecidas

In [ ]:
# Convertimos a Pandas para graficar
pdf = dfPy01.select("tasa_incidencia", "tasa_mortalidad").toPandas()
sns.set_theme(style="whitegrid")

# --- FIGURA DE BOXPLOTS ---
fig_box, axes_box = plt.subplots(1, 2, figsize=(15, 6))

sns.boxplot(x=pdf['tasa_incidencia'], ax=axes_box[0], color='skyblue')
axes_box[0].set_title('Boxplot de Tasa de Incidencia')

sns.boxplot(x=pdf['tasa_mortalidad'], ax=axes_box[1], color='salmon')
axes_box[1].set_title('Boxplot de Tasa de Mortalidad')

plt.tight_layout()
plt.show()

In [ ]:
# --- FIGURA DE HISTOGRAMAS ---
fig_hist, axes_hist = plt.subplots(1, 2, figsize=(15, 6))

sns.histplot(pdf['tasa_incidencia'], ax=axes_hist[0], color='skyblue')
axes_hist[0].set_title('Distribución de Tasa de Incidencia')

sns.histplot(pdf['tasa_mortalidad'], ax=axes_hist[1], color='salmon')
axes_hist[1].set_title('Distribución de Tasa de Mortalidad')

plt.tight_layout()
plt.show()

Se visualiza tanto en los boxplots como histogramas una muy clara asimetría positiva, donde los boxplots indican una gran cantidad de outliers, lo que indica con gran claridad que los datos no distribuyen normal. Pero al ser de naturaleza de tasas, estas tienden a tener una forma log-normal. Por lo que se verificara si este es el caso con estas bases

In [ ]:
# Aplicamos logaritmo a los datos para verificación de log-normalidad (evitando log(0) ya que existen valores cero)
incidencia_log = np.log(pdf['tasa_incidencia'].dropna() + 1e-9) 
mortalidad_log = np.log(pdf['tasa_mortalidad'].dropna() + 1e-9) 

# --- FIGURA DE HISTOGRAMAS LOG-NORMALES---
fig_hist, axes_hist = plt.subplots(1, 2, figsize=(15, 6))

sns.histplot(incidencia_log, ax=axes_hist[0], color='skyblue')
axes_hist[0].set_title('Distribución de Tasa de Incidencia')

sns.histplot(mortalidad_log, ax=axes_hist[1], color='salmon')
axes_hist[1].set_title('Distribución de Tasa de Mortalidad')

plt.tight_layout()
plt.show()

In [ ]:
# Dibujo de graficos QQ plot para visualización de normalidad
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# --- Q-Q Plot Incidencia ---
stats.probplot(incidencia_log, dist="norm", plot=ax1)
ax1.set_title('Q-Q Plot: Log(Tasa de Incidencia)')
ax1.get_lines()[0].set_markerfacecolor('skyblue') # Color de los puntos
ax1.get_lines()[0].set_alpha(0.5)

# --- Q-Q Plot Mortalidad ---
stats.probplot(mortalidad_log, dist="norm", plot=ax2)
ax2.set_title('Q-Q Plot: Log(Tasa de Mortalidad)')
ax2.get_lines()[0].set_markerfacecolor('salmon') # Color de los puntos
ax2.get_lines()[0].set_alpha(0.5)

plt.tight_layout()
plt.show()

In [ ]:
municipios_cero_incidencia = dfPy01.filter(dfPy01.tasa_incidencia == 0) \
    .select("municipio", "tasa_incidencia")

print(f"Municipios con incidencia cero: {municipios_cero_incidencia.count()}")
municipios_cero_incidencia.show()

municipios_cero_mortalidad = dfPy01.filter(dfPy01.tasa_mortalidad == 0) \
    .select("municipio", "tasa_mortalidad")

print(f"Municipios con mortalidad cero: {municipios_cero_mortalidad.count()}")
municipios_cero_mortalidad.show()

A pesar de aplicar logaritmo a las tasas, se puede ver tanto en los qqplots como en los diagramas de dispersión que no se da normalidad. Esto es dado por aquellos municipios con tasas 0, los cuales bajo el contexto son atípicos y no permiten la normalidad incluso bajo transformaciones, aunque también se evidencia que aplicar logaritmo ayuda a normalizar de mejor forma al grupo en general

In [ ]:
## Calculo de la correlación entre la tasa de incidencia y la tasa de mortalidad con Spark ML
# 1. Preparamos los datos eliminando nulos (necesario para scipy)
# Usamos el dataframe de Pandas 'pdf'
df_temp = pdf[['tasa_incidencia', 'tasa_mortalidad']].dropna()

# 2. Calculamos Spearman y el p-valor
coef_spearman, p_valor = stats.spearmanr(df_temp['tasa_incidencia'], df_temp['tasa_mortalidad'])

# 3. Mostrar resultados formateados
print("--- Prueba de Correlación de Spearman ---")
print(f"Coeficiente de correlación (rho): {coef_spearman:.4f}")
print(f"P-valor: {p_valor:.4e}") # Formato científico para p-valores pequeños

# 4. Interpretación automática
alpha = 0.05
if p_valor < alpha:
    print("\nInterpretación: La correlación es ESTADÍSTICAMENTE SIGNIFICATIVA.")
    print(f"Hay evidencia suficiente para rechazar la hipótesis nula (p < {alpha}).")
else:
    print("\nInterpretación: La correlación NO es estadísticamente significativa.")
    print("No hay evidencia suficiente para afirmar que existe una relación (p > 0.05).")

In [ ]:
# --- DIAGRAMA DE DISPERSIÓN (Relación) ---
plt.figure(figsize=(10, 6))
sns.regplot(data=pdf, x='tasa_incidencia', y='tasa_mortalidad',
            fit_reg=False, scatter_kws={'alpha':0.5})
plt.title(f'Dispersión: Incidencia vs Mortalidad\nSpearman: {coef_spearman:.2f} (p-val: {p_valor:.2e})')
plt.xlabel('Tasa de Incidencia (por 100k hab)')
plt.ylabel('Tasa de Mortalidad (por 1M hab)')
plt.show()

Como último análisis exploratorio, se visualiza que existe una correlación de 0.69 entre la tasa de mortalidad y la de incidencia, la cual al ser estadísticamente significativa indica que aquellos municipios con altas tasas de incidencia, tienden a tener también tasas de mortalidad altas, y también al caso contrario, municipios con tasas bajas de incidencia tienen bajas tasas de mortalidad.

## **Reporte de calidad**

In [ ]:
# Mostrar cantidad de nulos por columna
dfPy01.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) for c in dfPy01.columns]).show()

In [ ]:
# Mostrar aquellos municipios con valores nulos
# Filtramos filas donde la incidencia O la mortalidad sean nulas
municipios_con_nulos = dfPy01.filter(
    F.col("tasa_incidencia").isNull() | 
    F.isnan(F.col("tasa_incidencia")) |
    F.col("tasa_mortalidad").isNull() | 
    F.isnan(F.col("tasa_mortalidad"))
)

print(f"Total de municipios con datos faltantes: {municipios_con_nulos.count()}")
municipios_con_nulos.select("municipio", "tasa_incidencia", "tasa_mortalidad").show()

In [ ]:
print(f"Cantidad de municipios en el dataset: {dfPy01.count()}")

Se reportan dos municipios con datos nulos dentro de todo el dataset. Se evidencia que ninguno de estos dos municipios cuenta ni con tasas de incidencia ni con tasas de mortalidad, por este motivo no tenemos ningún tipo de información extra que nos pueda guiar a los valores reales de estos datos. Dado esto, y el hecho de que son solamente 2 municipios de los 1122, los cuales representan solo el 0.18% de los datos totales. Se decide eliminarlos para el análisis.

## **Planteamiento de preguntas investigativas**

1. ¿Es afectado el rendimiento en las pruebas ICFES por factores de riego de salud como pandemias y enfermedades?
2. ¿Pueden llegar a afectar altas tasas de mortalidad y/o incidencia de enfermedades como el COVID-19 por municipio, en el rendimiento de la población estudiantil de esa zona?

## **Limpieza, filtro y transformaciones iniciales**

In [ ]:
# Limpieza de valores faltantes
# Solo elimina la fila si falta la incidencia O la mortalidad
dfPy02 = dfPy01.dropna(subset=["tasa_incidencia", "tasa_mortalidad"])

In [ ]:
# Conteo de cantidad de municipios
print(f"Cantidad de municipios previo a la eliminación: {dfPy01.count()}")
print(f"Cantidad de municipios post a la eliminación: {dfPy02.count()}")

Dado los resultados obtenidos en el análisis exploratorio, se va a normalizar las tasas aplicándoles una transformación logarítmica y filtrando aquellos atípicos mencionados en el análisis y se guardara este nuevo conjunto de datos aparte, se realizara este paso, ya que en caso de ser necesario aplicar modelos con supuestos de normalidad, se esperaría un mejor resultado predictivo por medio de estos datos normalizados sobre los datos base. Aunque se mantendrá aún los datos base en caso de aplicar técnicas sin supuestos de normalidad o ser necesario dar mayor explicación a los modelos.

In [ ]:
# Filtramos: eliminamos nulos y valores que sean 0 en cualquiera de las dos tasas
df_positivos = dfPy01.filter(
    (F.col("tasa_incidencia") > 0) & 
    (F.col("tasa_mortalidad") > 0)
)

# Aplicamos la transformación logarítmica (Logaritmo Natural)
# Creamos nuevas columnas para no sobreescribir las originales
df_log = df_positivos.withColumn("log_incidencia", F.log("tasa_incidencia")) \
                     .withColumn("log_mortalidad", F.log("tasa_mortalidad"))
# Verificamos cuántos datos quedaron después del filtro
conteo_original = dfPy01.count()
conteo_final = df_log.count()

print(f"Filas originales: {conteo_original}")
print(f"Filas tras eliminar ceros y nulos: {conteo_final}")
print(f"Municipios descartados: {conteo_original - conteo_final}")

# 4. Mostramos una muestra del resultado
df_log.select("municipio", "tasa_incidencia", "log_incidencia", "tasa_mortalidad", "log_mortalidad").show(5)

# 5. Convertimos a Pandas para gráficas Q-Q
pdf_log = df_log.select("log_incidencia", "log_mortalidad").toPandas()

In [ ]:
#Graficación de QQ plot filtrado y transformado
#Configurar la figura
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# --- Q-Q Plot Incidencia ---
stats.probplot(pdf_log["log_incidencia"], dist="norm", plot=ax1)
ax1.set_title('Q-Q Plot: Log(Tasa de Incidencia)')
ax1.get_lines()[0].set_markerfacecolor('skyblue') # Color de los puntos
ax1.get_lines()[0].set_alpha(0.5)

# --- Q-Q Plot Mortalidad ---
stats.probplot(pdf_log["log_mortalidad"], dist="norm", plot=ax2)
ax2.set_title('Q-Q Plot: Log(Tasa de Mortalidad)')
ax2.get_lines()[0].set_markerfacecolor('salmon') # Color de los puntos
ax2.get_lines()[0].set_alpha(0.5)

plt.tight_layout()
plt.show()

Los gráficos QQ plot indican datos normalizados más suaves que los originales, y aunque no están perfectamente alineados, se ve una gran mejoría de normalidad que sobre los datos base. Lo que en modelos con supuestos de normalidad, estabilizaría en mayor cantidad las predicciones hechas, y por consiguiente, la calidad de las mismas.